# ROGII Wellbore Geology — v3 Clean Strategy + EDA

**Strategy pillars (leak-free, test-safe):**
| # | What | Why |
|---|------|-----|
| 1 | Drop formation surfaces (ANCC…BUDA) | Training-only columns — absent in test |
| 2 | `TVT_interp` — linear interp/extrap of TVT_input into gap | Smooth structural baseline |
| 3 | `TVT_geom` — per-well TVT ~ Z regression | Captures gross structural dip |
| 4 | `GR_type` + `GR_diff` from typewell at TVT_interp | Cheap proxy for log matching |
| 5 | `dist_to_gap_start`, `dist_to_gap_end` | Uncertainty grows toward gap center |
| 6 | Rolling GR mean/std (10, 20, 50, 100 ft) | Local log shape / formation boundaries |
| 7 | GroupKFold by well | Mimics truly unseen test wells |
| 8 | Final LightGBM on ALL gap rows | Maximum training signal |

In [ ]:
# %% [code] 1. Imports & Setup
import os, gc, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler
import joblib
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

INPUT  = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
WORK   = Path('/kaggle/working')
TRAIN  = INPUT / 'train'
TEST   = INPUT / 'test'

print('LightGBM version:', lgb.__version__)

In [ ]:
# %% [code] 2. Data Loading Functions
def load_well_data(base_dir):
    """Return list of (hw_df, tw_df, well_id) for all wells in directory."""
    base = Path(base_dir)
    wells = []
    for hw_path in sorted(base.glob('*__horizontal_well.csv')):
        well_id = hw_path.stem.replace('__horizontal_well', '')
        tw_path = base / f'{well_id}__typewell.csv'
        hw = pd.read_csv(hw_path)
        tw = pd.read_csv(tw_path) if tw_path.exists() else None
        hw['WELL'] = well_id
        if tw is not None:
            tw['WELL'] = well_id
        wells.append((hw, tw, well_id))
    return wells

train_wells = load_well_data(TRAIN)
test_wells = load_well_data(TEST)
print(f'Train wells: {len(train_wells)}, Test wells (example): {len(test_wells)}')

In [ ]:
# %% [code] 3. Core Feature Engineering – No Surface Columns (FIXED)

def sliding_gr_tvt(hw_gr, tw_tvt, tw_gr, window=20, step=5):
    """
    For each point in hw_gr, find best matching TVT by comparing
    the local GR window with typewell GR via correlation.
    Returns array of estimated TVT.
    """
    n = len(hw_gr)
    tvt_est = np.full(n, np.nan)
    half = window // 2
    
    tw_n = len(tw_gr)
    if tw_n < window:
        return tvt_est
    
    for i in range(n):
        i0 = max(0, i - half)
        i1 = min(n, i + half)
        hw_win = hw_gr[i0:i1]
        if len(hw_win) < 5:
            continue
        hw_win = np.nan_to_num(hw_win, nan=np.nanmean(hw_gr))
        
        best_corr = -2
        best_tvt = np.nan
        wlen = len(hw_win)
        
        for j in range(0, tw_n - wlen, step):
            tw_win = tw_gr[j:j+wlen]
            if np.std(tw_win) < 1e-9 or np.std(hw_win) < 1e-9:
                continue
            corr = np.corrcoef(hw_win, tw_win)[0,1]
            if corr > best_corr:
                best_corr = corr
                best_tvt = tw_tvt[j + wlen//2]
        tvt_est[i] = best_tvt
    
    return tvt_est

def add_features(hw_df, tw_df):
    """
    Build features for a single well. Only uses:
    - MD, X, Y, Z, GR, TVT_input
    - Typewell (TVT, GR)
    No formation surface columns.
    """
    df = hw_df.copy().reset_index(drop=True)
    n = len(df)
    
    # ----- Known TVT handling -----
    tvt_in = df['TVT_input'].values.astype(float)
    known_mask = ~np.isnan(tvt_in)
    
    # Forward/backward fill as trend anchor (do NOT use inside gap for training)
    tvt_filled = pd.Series(tvt_in).ffill().bfill().values
    df['TVT_anchor'] = tvt_filled
    
    # ----- Linear interpolation baseline -----
    if known_mask.sum() >= 2:
        known_md = df.loc[known_mask, 'MD']
        known_tvt = tvt_in[known_mask]
        f_interp = interp1d(known_md, known_tvt, kind='linear',
                            bounds_error=False, fill_value='extrapolate')
        df['TVT_interp'] = f_interp(df['MD'])
    else:
        df['TVT_interp'] = df['TVT_anchor']
    
    # ----- Dip feature (TVT ~ Z) -----
    if known_mask.sum() >= 2:
        from sklearn.linear_model import LinearRegression
        lr = LinearRegression()
        lr.fit(df.loc[known_mask, ['Z']], tvt_in[known_mask])
        df['TVT_geom'] = lr.predict(df[['Z']])
    else:
        df['TVT_geom'] = df['TVT_interp']
    
    # ----- Typewell GR alignment features -----
    if tw_df is not None and len(tw_df) > 5:
        tw = tw_df.dropna(subset=['TVT', 'GR']).sort_values('TVT')
        tw_tvt_arr = tw['TVT'].values
        tw_gr_arr  = tw['GR'].values
        
        # 1. Simple pointwise GR matching (fast)
        gr_to_tvt = interp1d(tw_gr_arr, tw_tvt_arr, kind='linear',
                             bounds_error=False, fill_value='extrapolate')
        df['TVT_from_GR_point'] = gr_to_tvt(df['GR'].values)
        
        # 2. Sliding window correlation feature (more robust)
        tvt_corr = sliding_gr_tvt(df['GR'].values, tw_tvt_arr, tw_gr_arr,
                                  window=30, step=3)
        df['TVT_from_GR_corr'] = tvt_corr
        
        # Fill missing correlation values with pointwise or anchor
        mask_nan = df['TVT_from_GR_corr'].isna()
        df.loc[mask_nan, 'TVT_from_GR_corr'] = df.loc[mask_nan, 'TVT_from_GR_point']
        # Still NaN -> use anchor
        df['TVT_from_GR_corr'] = df['TVT_from_GR_corr'].fillna(df['TVT_anchor'])
        
        # 3. Typewell GR mapped at anchor TVT (how does typewell GR compare?)
        tw_gr_func = interp1d(tw_tvt_arr, tw_gr_arr, kind='linear',
                              bounds_error=False, fill_value='extrapolate')
        df['TW_GR_at_anchor'] = tw_gr_func(df['TVT_anchor'])
        df['GR_diff'] = df['GR'] - df['TW_GR_at_anchor']
        
    else:
        # Fallback if no typewell
        df['TVT_from_GR_point'] = df['TVT_anchor']
        df['TVT_from_GR_corr'] = df['TVT_anchor']
        df['TW_GR_at_anchor'] = df['GR']
        df['GR_diff'] = 0.0
    
    # ----- Distance to known ends (robust version) -----
    eval_mask = ~known_mask
    if eval_mask.any():
        gap_start = np.argmax(eval_mask)                # first True
        gap_end   = len(eval_mask) - 1 - np.argmax(eval_mask[::-1])  # last True
        df['dist_gap_start'] = np.abs(np.arange(n) - gap_start)
        df['dist_gap_end']   = np.abs(np.arange(n) - gap_end)
    else:
        df['dist_gap_start'] = 0
        df['dist_gap_end'] = 0
    
    # ----- GR rolling statistics -----
    gr = df['GR'].values
    for w in [5, 11, 21, 51]:
        df[f'GR_mean_{w}'] = pd.Series(gr).rolling(w, center=True, min_periods=1).mean().values
        df[f'GR_std_{w}']  = pd.Series(gr).rolling(w, center=True, min_periods=1).std().fillna(0).values
    
    df['GR_grad'] = np.gradient(np.nan_to_num(gr))
    
    # ----- Trajectory derived -----
    df['dZ_dMD'] = np.gradient(df['Z']) / np.gradient(df['MD']).clip(1e-6)
    df['cum_MD'] = df['MD'] - df['MD'].min()
    df['MD_norm'] = (df['MD'] - df['MD'].min()) / (df['MD'].max() - df['MD'].min() + 1e-9)
    
    # ----- Row index -----
    df['row_idx'] = np.arange(n)
    df['row_rev'] = n - 1 - np.arange(n)
    
    return df

In [ ]:
# %% [code] 4. Build Training Feature Matrix
# We only train on rows where TVT is known (all rows in training have TVT).
train_parts = []
well_labels = []

print('Building training feature matrix...')
t0 = time.time()
for hw, tw, wname in train_wells:
    feat = add_features(hw, tw)
    # Keep all rows, target is TVT (present in training)
    train_parts.append(feat)
    well_labels.extend([wname] * len(feat))

train_all = pd.concat(train_parts, ignore_index=True)
print(f'Total train rows: {len(train_all):,} in {time.time()-t0:.1f}s')

# Define feature columns – exclude identifiers, target, and raw coordinates
drop_cols = ['WELL', 'TVT', 'TVT_input', 'MD', 'X', 'Y', 'Z',
             'GR', 'TVT_anchor']  # we keep some, drop originals that could leak
# Also drop any remaining surface columns if they accidentally appeared
for c in train_all.columns:
    if c.upper() in ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']:
        drop_cols.append(c)

feature_cols = [c for c in train_all.columns if c not in drop_cols
                and train_all[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
print(f'Feature count: {len(feature_cols)}')
print(feature_cols[:20], '...')

X = train_all[feature_cols].values
y = train_all['TVT'].values
groups = np.array(well_labels)

In [ ]:
# %% [code] 5. LightGBM with GroupKFold
PARAMS = dict(
    objective='regression',
    metric='rmse',
    learning_rate=0.05,
    num_leaves=255,
    min_child_samples=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    lambda_l1=0.1,
    lambda_l2=0.1,
    n_estimators=3000,
    device='gpu',
    gpu_platform_id=0,
    gpu_device_id=0,
    verbose=-1,
    random_state=SEED,
    n_jobs=-1,
)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
oof_preds_lgb = np.zeros(len(X))
models_lgb = []
fold_rmses = []

print('Training LightGBM...')
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_cols)
    dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    
    model = lgb.train(
        PARAMS,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(period=200)]
    )
    oof_preds_lgb[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_rmse = np.sqrt(mean_squared_error(y_va, oof_preds_lgb[va_idx]))
    fold_rmses.append(fold_rmse)
    models_lgb.append(model)
    print(f'Fold {fold+1} RMSE: {fold_rmse:.4f} | Best iter: {model.best_iteration}')

oof_rmse_lgb = np.sqrt(mean_squared_error(y, oof_preds_lgb))
print(f'\nLGB OOF RMSE: {oof_rmse_lgb:.4f} | Fold RMSEs: {[f"{v:.4f}" for v in fold_rmses]}')

In [ ]:
# %% [code] 6. Train a Small MLP for Ensemble (on filtered features)
# Use a subset of the top LGB features for the MLP (faster, avoids overfit)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# Get top features from LGB importance
importances = np.zeros(len(feature_cols))
for m in models_lgb:
    importances += m.feature_importance(importance_type='gain')
importances /= len(models_lgb)
top_idx = np.argsort(importances)[-50:]  # top 50 features
top_features = [feature_cols[i] for i in top_idx]
X_mlp = train_all[top_features].values

# Standardize
scaler = StandardScaler()
X_mlp_scaled = scaler.fit_transform(X_mlp)

oof_preds_mlp = np.zeros(len(X))
models_mlp = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, X_va = X_mlp_scaled[tr_idx], X_mlp_scaled[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    mlp = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu',
                       alpha=0.001, batch_size=2048, max_iter=200,
                       early_stopping=True, validation_fraction=0.1,
                       random_state=SEED, verbose=False)
    mlp.fit(X_tr, y_tr)
    oof_preds_mlp[va_idx] = mlp.predict(X_va)
    models_mlp.append(mlp)

oof_rmse_mlp = np.sqrt(mean_squared_error(y, oof_preds_mlp))
print(f'MLP OOF RMSE: {oof_rmse_mlp:.4f}')

# Simple weighted blend (weights chosen on OOF)
blend_weight = 0.8  # more weight on LGB if it's better
oof_blend = blend_weight * oof_preds_lgb + (1 - blend_weight) * oof_preds_mlp
print(f'Blended OOF RMSE: {np.sqrt(mean_squared_error(y, oof_blend)):.4f}')

In [ ]:
# %% [code] 7. Train Final Models on Full Data & Prepare Test Predictor
# Retrain LGB on all training data
lgb_final = lgb.train(
    PARAMS,
    lgb.Dataset(X, label=y, feature_name=feature_cols),
    num_boost_round=3000,
    valid_sets=[lgb.Dataset(X, label=y, feature_name=feature_cols)],
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

# Retrain MLP on all data
scaler_final = StandardScaler()
X_mlp_final = scaler_final.fit_transform(X_mlp)
mlp_final = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu',
                         alpha=0.001, batch_size=2048, max_iter=200,
                         random_state=SEED)
mlp_final.fit(X_mlp_final, y)

# Function to predict a test well
def predict_test_well(hw, tw):
    feat = add_features(hw, tw)
    X_f = feat[feature_cols].values
    # LGB prediction
    pred_l = lgb_final.predict(X_f, num_iteration=lgb_final.best_iteration)
    # MLP prediction
    X_mlp_f = feat[top_features].values
    X_mlp_f = scaler_final.transform(X_mlp_f)
    pred_m = mlp_final.predict(X_mlp_f)
    
    # Blend
    pred = blend_weight * pred_l + (1 - blend_weight) * pred_m
    
    # Replace known TVT with exact values where no gap (safety)
    tvt_in = hw['TVT_input'].values.astype(float)
    known = ~np.isnan(tvt_in)
    pred[known] = tvt_in[known]
    
    # Optional: Savitzky-Golay smoothing on evaluation zone
    eval_mask = np.isnan(tvt_in)
    if eval_mask.sum() > 5:
        zone = pred[eval_mask]
        win = min(21, len(zone) if len(zone) % 2 == 1 else len(zone) - 1)
        win = max(3, win)
        if win % 2 == 0: win -= 1
        try:
            smoothed = savgol_filter(zone, window_length=win, polyorder=3)
            pred[eval_mask] = smoothed
        except:
            pass
    return pred

In [ ]:
# %% [code] 8. Generate Submission
# Load sample submission to get required IDs
sub = pd.read_csv(INPUT / 'sample_submission.csv')
sub[['well_id', 'row_idx']] = sub['id'].str.rsplit('_', n=1, expand=True)
sub['row_idx'] = sub['row_idx'].astype(int)

# Build dictionary of test wells
test_dict = {w: (hw, tw) for hw, tw, w in test_wells}

predictions = {}
for wid, (hw, tw) in test_dict.items():
    preds = predict_test_well(hw, tw)
    predictions[wid] = preds

# Fill submission
tvt_values = []
for _, row in sub.iterrows():
    wid = row['well_id']
    ridx = row['row_idx']
    if wid in predictions:
        arr = predictions[wid]
        if ridx < len(arr):
            tvt_values.append(float(arr[ridx]))
        else:
            tvt_values.append(float(np.nanmean(arr)))
    else:
        tvt_values.append(0.0)

sub['tvt'] = tvt_values
final_sub = sub[['id', 'tvt']]
final_sub.to_csv(WORK / 'submission.csv', index=False)

print('Submission saved. Shape:', final_sub.shape)
print(final_sub.head())

In [ ]:
# %% [code] 9. Quick Validation Output
print(f'LGB OOF RMSE: {oof_rmse_lgb:.4f}')
print(f'MLP OOF RMSE: {oof_rmse_mlp:.4f}')
print(f'Blended OOF: {np.sqrt(mean_squared_error(y, oof_blend)):.4f}')